In [3]:
import os
import re
import pandas as pd
import nibabel as nib
import numpy as np
from scipy.ndimage import label, center_of_mass, sum_labels
from tqdm.auto import tqdm

print("--- Step 4.2: Brain Metastases Quantification ---")

# --- 1. パス設定 (あなたのローカル環境に合わせてパスを修正済み) ---
# **入力フォルダ：セグメンテーションマスクが保存されている場所**
seg_dir = '/home/masaki/mystudy/my_yale_data/yale_brain_mets_segmentations' 
# **出力ファイル：計算結果を保存するCSVファイルの場所と名前**
output_csv_path = '/home/masaki/mystudy/my_yale_data/brain_mets_quantification_final.csv' 

# --- 2. データ処理の準備 ---
quantification_results = []
# フォルダ内の.nii.gzファイルをリストアップ
nii_files = [f for f in os.listdir(seg_dir) if f.endswith('.nii.gz')]

file_pattern = re.compile(
    r'^(?P<patient_id>[A-Z0-9]+_[A-Z0-9]+)_'
    r'(?P<scan_date>\d{4}-\d{2}-\d{2})_'
    r'(?P<scan_time>\d{2}-\d{2}-\d{2})\.nii\.gz$'
)

print(f"Found {len(nii_files)} segmentation files to process.")

# --- 3. 各ファイルをループ処理して定量化 ---
for filename in tqdm(nii_files, desc="Processing Files"):
    match = file_pattern.match(filename)
    if not match:
        continue

    patient_id = match.group('patient_id')
    scan_date = match.group('scan_date')
    scan_time = match.group('scan_time').replace('-', ':')
    scan_datetime = f"{scan_date} {scan_time}"
    filepath = os.path.join(seg_dir, filename)

    try:
        nii_image = nib.load(filepath)
        segmentation_data = nii_image.get_fdata()
        voxel_dims = nii_image.header.get_zooms()
        voxel_volume_mm3 = np.prod(voxel_dims)
        
        total_volume_mm3 = np.sum(segmentation_data > 0) * voxel_volume_mm3
        
        labeled_mask, num_features = label(segmentation_data > 0)
        lesion_count = num_features
        
        centroids = []
        individual_volumes_mm3 = []
        
        if lesion_count > 0:
            labels = np.arange(1, lesion_count + 1)

            # c) 各腫瘍の重心を計算
            centroids = center_of_mass(segmentation_data, labeled_mask, labels)
            # `center_of_mass`は病変が1つの場合タプルを直接返すため、リストに変換する
            if not isinstance(centroids, list):
                centroids = [centroids]
            centroids = [tuple(map(float, c)) for c in centroids]

            # d) 個々の腫瘍の体積を計算
            individual_volumes_voxels = sum_labels(segmentation_data > 0, labeled_mask, labels)
            # `sum_labels`も病変が1つの場合、リストではなく数値を返すことがあるため対応
            if not isinstance(individual_volumes_voxels, (list, np.ndarray)):
                individual_volumes_voxels = [individual_volumes_voxels]

            individual_volumes_mm3 = [float(vol * voxel_volume_mm3) for vol in individual_volumes_voxels]

        quantification_results.append({
            'FileName': filename,
            'PatientID': patient_id,
            'ScanDateTime': scan_datetime,
            'TotalVolume_mm3': total_volume_mm3,
            'LesionCount': lesion_count,
            'Centroids_vox': str(centroids),
            'IndividualVolumes_mm3': str(individual_volumes_mm3)
        })

    except Exception as e:
        print(f"Error processing file {filename}: {e}")

# --- 4. 結果をPandas DataFrameに変換してCSVに保存 ---
df_results = pd.DataFrame(quantification_results)
df_results.to_csv(output_csv_path, index=False)

print(f"\n✅ Quantification complete!")
print(f"Results saved to: {output_csv_path}")
print("\n--- First 5 rows of the final result ---")
print(df_results.head())

/root/anaconda3/envs/mymodel/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- Step 4.2: Brain Metastases Quantification ---
Found 8996 segmentation files to process.


Processing Files: 100%|███████████████████████████████████████| 8996/8996 [1:45:38<00:00,  1.42it/s]


✅ Quantification complete!
Results saved to: /home/masaki/mystudy/my_yale_data/brain_mets_quantification_final.csv

--- First 5 rows of the final result ---
                                     FileName        PatientID  \
0  YG_XDBSZJORNHS3_2018-06-22_09-12-49.nii.gz  YG_XDBSZJORNHS3   
1  YG_H19ZB9NMLHLJ_2015-12-17_07-19-13.nii.gz  YG_H19ZB9NMLHLJ   
2  YG_ZBN9ZQUDDHW1_2018-05-26_11-47-27.nii.gz  YG_ZBN9ZQUDDHW1   
3  YG_G3R2RIQS69WL_2014-12-05_13-09-30.nii.gz  YG_G3R2RIQS69WL   
4  YG_PFHUD3ZAEUJP_2010-12-31_15-28-51.nii.gz  YG_PFHUD3ZAEUJP   

          ScanDateTime  TotalVolume_mm3  LesionCount  \
0  2018-06-22 09:12:49      1630.000000            4   
1  2015-12-17 07:19:13      7238.731929            2   
2  2018-05-26 11:47:27      1136.000000            6   
3  2014-12-05 13:09:30         0.000000            0   
4  2010-12-31 15:28:51      1313.984773            5   

                                       Centroids_vox  \
0  [(88.14285714285714, 126.28571428571429, 56.76...

In [4]:
import pandas as pd
import numpy as np
import ast
from tqdm import tqdm

print("--- Final Step 5: Generating 2-Category Labeled Dataset ---")

# ===================================================================
# 1. ステップ4で作成した、詳細な数値データを読み込む
# ===================================================================
print("\n[1] Loading detailed quantification data...")
csv_path = '/home/masaki/mystudy/my_yale_data/brain_mets_quantification_final.csv' 
df = pd.read_csv(csv_path)

# 文字列をリストに変換
tqdm.pandas(desc="Parsing Centroids")
df['Centroids_vox'] = df['Centroids_vox'].progress_apply(ast.literal_eval)
tqdm.pandas(desc="Parsing Individual Volumes")
df['IndividualVolumes_mm3'] = df['IndividualVolumes_mm3'].progress_apply(ast.literal_eval)

df['ScanDateTime'] = pd.to_datetime(df['ScanDateTime'])
df = df.sort_values(by=['PatientID', 'ScanDateTime']).reset_index(drop=True)
print("✅ Data loading complete.")

# ===================================================================
# 2. RANO-BM基準に基づいた詳細なラベル付け（これは中間処理として必要）
# ===================================================================
print("\n[2] Applying detailed RANO-BM rules (as an intermediate step)...")
# ベースライン情報を結合
baseline_df = df.loc[df.groupby('PatientID')['ScanDateTime'].idxmin()].copy()
baseline_df = baseline_df[['PatientID', 'TotalVolume_mm3', 'Centroids_vox']]
baseline_df.rename(columns={'TotalVolume_mm3': 'BaselineVolume', 'Centroids_vox': 'BaselineCentroids'}, inplace=True)
df_merged = pd.merge(df, baseline_df, on='PatientID', how='left')

# 新規病変を判定
def count_new_lesions(baseline_centroids, followup_centroids, distance_threshold=15.0):
    if not baseline_centroids: return len(followup_centroids)
    if not followup_centroids: return 0
    new_lesion_count = 0
    for f_centroid in followup_centroids:
        is_new = True
        for b_centroid in baseline_centroids:
            if np.linalg.norm(np.array(f_centroid) - np.array(b_centroid)) < distance_threshold:
                is_new = False
                break
        if is_new: new_lesion_count += 1
    return new_lesion_count

tqdm.pandas(desc="Detecting New Lesions")
df_merged['NewLesionCount'] = df_merged.progress_apply(lambda row: count_new_lesions(row['BaselineCentroids'], row['Centroids_vox']), axis=1)

# 詳細なパターンラベルを付ける
def assign_detailed_pattern(row, patient_baselines):
    baseline_time = patient_baselines.get(row['PatientID'])
    if row['ScanDateTime'] == baseline_time: return 'Baseline'
    baseline_volume = row['BaselineVolume']
    if baseline_volume < 1e-6: return 'Progression_NewLesion' if row['TotalVolume_mm3'] > 1e-6 else 'Stable_Disease'
    volume_change_rate = (row['TotalVolume_mm3'] - baseline_volume) / baseline_volume
    if row['NewLesionCount'] > 0: return 'Progression_NewLesion'
    if volume_change_rate >= 0.20: return 'Progression_Growth'
    if volume_change_rate <= -0.30: return 'Partial_Response'
    return 'Stable_Disease'

patient_baselines = df.loc[df.groupby('PatientID')['ScanDateTime'].idxmin()].set_index('PatientID')['ScanDateTime'].to_dict()
tqdm.pandas(desc="Assigning Detailed Labels")
df_merged['DetailedPattern'] = df_merged.progress_apply(lambda row: assign_detailed_pattern(row, patient_baselines), axis=1)
print("✅ Detailed labeling complete.")

# ===================================================================
# 3. 最終的な2カテゴリーへの統合
# ===================================================================
print("\n[3] Simplifying labels into the final 2 categories...")
def simplify_pattern(pattern):
    if pattern in ['Progression_NewLesion', 'Progression_Growth']:
        return 'Progression'
    elif pattern in ['Partial_Response', 'Stable_Disease']:
        return 'Favorable_Outcome'
    else:
        return pattern # 'Baseline' などはそのまま

# AI学習用の最終的なラベル列 'FinalPattern' を作成
df_merged['FinalPattern'] = df_merged['DetailedPattern'].apply(simplify_pattern)
print("✅ Simplification complete.")

# ===================================================================
# 4. 最終データセットの保存と確認
# ===================================================================
print("\n[4] Saving the final dataset for AI training...")
final_csv_path = '/home/masaki/mystudy/my_yale_data/final_labeled_data.csv' 
df_merged.to_csv(final_csv_path, index=False)

print(f"\n🚀 All set! Your final dataset is ready for AI training!")
print(f"Saved to: {final_csv_path}")

print("\n--- Final Data Distribution ---")
print(df_merged['FinalPattern'].value_counts())

--- Final Step 5: Generating 2-Category Labeled Dataset ---

[1] Loading detailed quantification data...


Parsing Individual Volumes: 100%|████████████████████████████| 8996/8996 [00:00<00:00, 77911.43it/s]


✅ Data loading complete.

[2] Applying detailed RANO-BM rules (as an intermediate step)...


Assigning Detailed Labels: 100%|█████████████████████████████| 8996/8996 [00:00<00:00, 95160.17it/s]


✅ Detailed labeling complete.

[3] Simplifying labels into the final 2 categories...
✅ Simplification complete.

[4] Saving the final dataset for AI training...

🚀 All set! Your final dataset is ready for AI training!
Saved to: /home/masaki/mystudy/my_yale_data/final_labeled_data.csv

--- Final Data Distribution ---
FinalPattern
Progression          7258
Baseline             1362
Favorable_Outcome     376
Name: count, dtype: int64
